Создать сеть на базе LSTM используя TensorFlow (Keras). Сеть должна принимать на вход текстовый файл и на его базе генерировать свою абракадабру. Отчет должен содержать кроме кода, обучающий файл и результат генерации.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Activation
from tensorflow.keras.optimizers import RMSprop
import random
from tqdm import tqdm

In [ ]:
filename = 'test.txt'
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read().lower()

print(len(text))

In [ ]:
chars = sorted(list(set(text)))
char_indices = dict((c, i) for i, c in enumerate(chars))
indices_char = dict((i, c) for i, c in enumerate(chars))
vocab_size = len(chars)

maxlen = 40
step = 3     
sentences = []
next_chars = []
#формируем обучающие примеры
for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i: i + maxlen])
    next_chars.append(text[i + maxlen])

print(len(sentences))
#One-hot encoding – это представление категориальных данных (в данном случае символов) в виде бинарных векторов фиксированной длины, равной числу категорий
X = np.zeros((len(sentences), maxlen, vocab_size), dtype=bool)
y = np.zeros((len(sentences), vocab_size), dtype=bool)

for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        X[i, t, char_indices[char]] = 1
    y[i, char_indices[next_chars[i]]] = 1

In [ ]:
model = Sequential()
model.add(LSTM(128, input_shape=(maxlen, vocab_size)))
model.add(Dense(vocab_size)) #полносвязный слой
model.add(Activation('softmax')) # категориальная кросс-энтропия. Она измеряет расхождение между двумя распределениями: предсказанным (y_pred, выход softmax) и истинным (y_true, one-hot метка).
optimizer = RMSprop(learning_rate=0.01)
model.compile(loss='categorical_crossentropy', optimizer=optimizer)

In [ ]:
def sample(preds, temperature=1.0):
    preds = np.asarray(preds).astype('float64')
    preds = np.log(preds + 1e-7) / temperature 
    exp_preds = np.exp(preds)
    preds = exp_preds / np.sum(exp_preds)
    probas = np.random.multinomial(1, preds, 1)
    return np.argmax(probas)

def generate_text(length=400, temperature=0.5):
    start_index = random.randint(0, len(text) - maxlen - 1)
    generated = ''
    sentence = text[start_index: start_index + maxlen]
    generated += sentence
    result_text = sentence
    
    for i in range(length):
        x_pred = np.zeros((1, maxlen, vocab_size))
        for t, char in enumerate(sentence):
            x_pred[0, t, char_indices[char]] = 1.

        preds = model.predict(x_pred, verbose=0)[0]
        next_index = sample(preds, temperature)
        next_char = indices_char[next_index]

        sentence = sentence[1:] + next_char
        result_text += next_char

    return result_text

In [ ]:
final_text = generate_text(length=500, temperature=0.7)
print(final_text)

In [ ]:
final_text = generate_text(length=500, temperature=0.7)
print(final_text)

![image.png](attachment:10c98cd1-4b0e-4d7e-8b54-6433f019643f.png)